In [91]:
from shutil import rmtree
from pathlib import Path

import matplotlib.pyplot as plt

from vot_utils.data import DATA_DIRECTORY, RESULTS_DIRECTORY

In [92]:
import torch

from torch import is_tensor
from torch import tensor
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader

In [93]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [94]:
from functools import partial

from references.transfusion_pytorch.transfusion_pytorch.transfusion import (
    Attention,
    FeedForward,
    Transformer,
    default,
    exists,
    cast_tuple,
    default_to_modality_shape_fn,
    identity,
    add_temp_batch_dim,
    char_tokenize,
    decode_chars,
    append_dims,
    ModalityInfo,
    get_model_output_to_flow_fn,
    pack_one_with_inverse,
    divisible_by,
    eval_decorator,
    rearrange,
    repeat,
    typecheck
)

import torch.nn.functional as F

from torch import nn
from torch.nn import ModuleList
from torch.nn import Linear

from axial_positional_embedding import ContinuousAxialPositionalEmbedding
from rotary_embedding_torch import RotaryEmbedding, apply_rotary_emb
from torchdiffeq import odeint
from ema_pytorch import EMA

from beartype import beartype
from beartype.door import is_bearable

In [123]:
class Transfusion(nn.Module):

    def __init__(
        self,
        num_text_tokens,
        transformer,
        dim_latent=None,
        model_output_clean=True,
        channel_first_latent=False,
        modality_default_shape=None,
        modality_encoder=None,
        modality_decoder=None,
        add_pos_emb=False,
        modality_num_dim=None,
        velocity_consistency_loss_weight=0.1,
        reconstruction_loss_weight=0,
        to_modality_shape_fn=default_to_modality_shape_fn,
        fallback_to_default_shape_if_invalid=False,
        modality_encoder_decoder_requires_batch_dim=True,
        pre_post_transformer_enc_dec=None,
        ignore_index=-1,
        flow_loss_weight=1.0,
        text_loss_weight=1.0,
        odeint_kwargs=dict(atol=1e-5, rtol=1e-5, method="midpoint"),
        eps=1e-2,
        prob_uncond=0.1,
    ):
        super().__init__()

        if isinstance(transformer, dict):
            transformer = Transformer(**transformer).to(device)

        self.transformer = transformer

        self.dim = transformer.dim
        dim_latent = default(dim_latent, self.dim)

        self.dim_latents = cast_tuple(dim_latent)

        self.num_modalities = len(self.dim_latents)
        self.channel_first_latent = cast_tuple(
            channel_first_latent, self.num_modalities
        )
        self.to_modality_shape_fn = cast_tuple(
            to_modality_shape_fn, self.num_modalities
        )

        if not exists(modality_default_shape) or is_bearable(
            modality_default_shape, tuple[int, ...]
        ):
            modality_default_shape = (modality_default_shape,) * self.num_modalities

        self.modality_default_shape = modality_default_shape

        self.modality_num_dim = cast_tuple(modality_num_dim, self.num_modalities)
        self.add_pos_emb = cast_tuple(add_pos_emb, self.num_modalities)

        self.pos_emb_mlp = ModuleList([])

        for modality_add_pos_emb, modality_ndim in zip(
            self.add_pos_emb, self.modality_num_dim
        ):
            if not modality_add_pos_emb:
                self.pos_emb_mlp.append(None)
                continue

            pos_generating_mlp = ContinuousAxialPositionalEmbedding(
                dim=self.dim,
                num_axial_dims=modality_ndim,
            )

            self.pos_emb_mlp.append(pos_generating_mlp)

        modality_encoder = cast_tuple(
            modality_encoder, 1 if exists(modality_encoder) else self.num_modalities
        )
        modality_decoder = cast_tuple(
            modality_decoder, 1 if exists(modality_decoder) else self.num_modalities
        )

        self.modality_encoder = ModuleList(modality_encoder)
        self.modality_decoder = ModuleList(modality_decoder)

        self.maybe_add_temp_batch_dim = (
            add_temp_batch_dim
            if modality_encoder_decoder_requires_batch_dim
            else identity
        )

        self.num_text_tokens = num_text_tokens

        num_text_special_ids = 3
        self.sos_id, self.eos_id, self.null_text_id = (
            num_text_tokens,
            (num_text_tokens + 1),
            (num_text_tokens + 2),
        )

        num_modality_special_ids = self.num_modalities * 2
        som_eom_tensor = (
            torch.arange(num_modality_special_ids)
            + num_text_tokens
            + num_text_special_ids
        )
        som_eom_tensor = rearrange(
            som_eom_tensor, "(start_end m) -> start_end m", start_end=2
        )
        self.som_ids, self.eom_ids = som_eom_tensor.tolist()

        meta_token_offset = (
            num_text_tokens + num_text_special_ids + num_modality_special_ids
        )
        self.meta_id = meta_token_offset
        num_meta_tokens = 128 + 1

        self.char_tokenizer = partial(char_tokenize, offset=meta_token_offset + 1)
        self.decode_chars = partial(decode_chars, offset=meta_token_offset + 1)

        pre_post_transformer_enc_dec = cast_tuple(
            pre_post_transformer_enc_dec, self.num_modalities
        )

        latent_to_model_projs = []
        model_to_latent_projs = []

        for (
            dim_latent,
            enc_dec,
        ) in zip(self.dim_latents, pre_post_transformer_enc_dec):
            pre_attend_enc, post_attend_dec = default(enc_dec, (None, None))

            latent_to_model_proj = (
                Linear(dim_latent, self.dim)
                if dim_latent != self.dim
                else nn.Identity()
            )
            model_to_latent_proj = Linear(self.dim, dim_latent, bias=False)

            latent_to_model_projs.append(default(pre_attend_enc, latent_to_model_proj))
            model_to_latent_projs.append(default(post_attend_dec, model_to_latent_proj))

        self.latent_to_model_projs = ModuleList(latent_to_model_projs)
        self.model_to_latent_projs = ModuleList(model_to_latent_projs)

        self.rotary_emb = RotaryEmbedding(transformer.dim_head)

        effective_num_text_tokens = (
            num_text_tokens
            + num_text_special_ids
            + num_modality_special_ids
            + num_meta_tokens
        )

        self.text_embed = nn.Embedding(effective_num_text_tokens, self.dim)
        self.to_text_logits = Linear(self.dim, effective_num_text_tokens, bias=False)
        text_only_mask = torch.arange(effective_num_text_tokens) < num_text_tokens
        self.register_buffer("text_only_logits_mask", text_only_mask, persistent=False)

        self.ignore_index = ignore_index
        self.flow_loss_weight = flow_loss_weight
        self.text_loss_weight = text_loss_weight
        self.velocity_consistency_loss_weight = velocity_consistency_loss_weight
        self.has_recon_loss = reconstruction_loss_weight > 0.0
        self.reconstruction_loss_weight = reconstruction_loss_weight
        self.model_output_clean = model_output_clean
        self.eps = eps
        self.odeint_fn = partial(odeint, **odeint_kwargs)
        self.prob_uncond = prob_uncond
        self.register_buffer("zero", tensor(0.0), persistent=False)

    @property
    def device(self):
        return next(self.parameters()).device

    def create_ema(self, beta=0.99, *ema_kwargs) -> EMA:

        ema = EMA(self, beta=beta, forward_method_names=("generate_modality_only",))

        return ema

    def muon_parameters(self):
        params = []

        for m in self.modules():
            if isinstance(m, Attention):
                params.extend(
                    [
                        *m.to_v.parameters(),
                        *m.to_out.parameters(),
                    ]
                )
            elif isinstance(m, FeedForward):
                params.extend([m.net[0].weight, m.net[-1].weight])

        return params

    def get_modality_info(self, modality_type):
        modality_type = default(modality_type, 0)
        return ModalityInfo(
            encoder=self.modality_encoder[modality_type],
            decoder=self.modality_decoder[modality_type],
            latent_to_model=self.latent_to_model_projs[modality_type].to(device),
            model_to_latent=self.model_to_latent_projs[modality_type].to(device),
            add_pos_emb=self.add_pos_emb[modality_type],
            pos_emb_mlp=self.pos_emb_mlp[modality_type].to(device),
            num_dim=self.modality_num_dim[modality_type],
            dim_latent=self.dim_latents[modality_type],
            default_shape=self.modality_default_shape[modality_type],
            som_id=self.som_ids[modality_type],
            eom_id=self.eom_ids[modality_type],
            to_shape_fn=self.to_modality_shape_fn[modality_type],
            channel_first_latent=self.channel_first_latent[modality_type],
            modality_type=modality_type,
        )

    @torch.no_grad()
    @eval_decorator
    @typecheck
    def generate_modality_only(
        self,
        batch_size=1,
        modality_type=None,
        fixed_modality_shape=None,
        modality_steps=16,
    ):
        device = self.device
        mod = self.get_modality_info(modality_type)

        modality_shape = default(fixed_modality_shape, mod.default_shape)
        noise = torch.randn(
            (batch_size, *modality_shape, mod.dim_latent), device=device
        )

        if mod.channel_first_latent:
            noise = rearrange(noise, "b ... d -> b d ...")

        def ode_step_fn(step_times, denoised):
            step_times = repeat(step_times, " -> b", b=batch_size)

            flow = self.forward(
                denoised,
                times=step_times,
                modality_type=modality_type,
                encode_modality=False,
                return_loss=False,
            )

            return flow

        times = torch.linspace(0.0, 1.0, modality_steps, device=device)
        trajectory = self.odeint_fn(ode_step_fn, noise, times)

        # add the sampled modality tokens

        sampled_modality = trajectory[-1]

        # decode

        if exists(mod.decoder):
            mod.decoder.eval()
            sampled_modality = mod.decoder(sampled_modality)

        return sampled_modality

    def forward(
        self,
        modalities,
        velocity_consistency_ema_model=None,
        velocity_consistency_delta_time=1e-5,
        modality_type=None,
        times=None,
        encode_modality=True,
        return_loss=True,
    ):
        if isinstance(velocity_consistency_ema_model, EMA):
            velocity_consistency_ema_model = velocity_consistency_ema_model.ema_model

        requires_velocity_consistency = exists(velocity_consistency_ema_model)

        modalities = modalities.to(self.device)
        orig_modalities = modalities

        modality_type = default(modality_type, 0)
        mod = self.get_modality_info(modality_type)

        if encode_modality and exists(mod.encoder):
            with torch.no_grad():
                mod.encoder.eval()
                modalities = mod.encoder(modalities).detach()

        tokens = modalities
        batch, device = tokens.shape[0], tokens.device

        if not exists(times):
            times = torch.rand((batch,), device=device)

        if requires_velocity_consistency:
            orig_times = times.clone()
            times = times * (
                1.0 - velocity_consistency_delta_time
            )  # make sure times are max of 1. - small delta, for velocity consistency

        padded_times = append_dims(times, tokens.ndim - 1)
        noise = torch.randn_like(tokens)
        noised_tokens = padded_times * tokens + (1.0 - padded_times) * noise
        flow = tokens - noise

        model_output_to_flow = (
            get_model_output_to_flow_fn(noised_tokens, times, self.eps)
            if self.model_output_clean
            else identity
        )

        noised_tokens = mod.latent_to_model(noised_tokens)

        if mod.add_pos_emb:
            _, *axial_dims, _ = noised_tokens.shape

        noised_tokens, inverse_pack_axial_dims = pack_one_with_inverse(
            noised_tokens, "b * d"
        )

        if mod.add_pos_emb:
            axial_pos_emb = mod.pos_emb_mlp(tensor(axial_dims), flatten=True)
            noised_tokens = noised_tokens + axial_pos_emb

        embed = self.transformer(
            noised_tokens,
            times=times,
            modality_only=True,
        )

        embed = inverse_pack_axial_dims(embed)
        model_output = mod.model_to_latent(embed)
        pred_flow = model_output_to_flow(model_output)

        if not return_loss:
            return pred_flow

        flow_loss = F.mse_loss(pred_flow, flow)

        # maybe velocity consistency loss

        velocity_loss = 0

        if requires_velocity_consistency:

            with torch.no_grad():
                flow_with_delta_time = velocity_consistency_ema_model.forward(
                    modalities=modalities,
                    modality_type=modality_type,
                    times=orig_times + velocity_consistency_delta_time,
                    encode_modality=False,  # modality already encoded
                    return_loss=False,
                )

            velocity_loss = F.mse_loss(flow, flow_with_delta_time)

        if self.has_recon_loss:
            assert encode_modality

            recon = noise + pred_flow * (1.0 - padded_times)

            if exists(mod.decoder):
                with torch.no_grad():
                    mod.decoder.eval()
                    recon = mod.decoder(recon)

            recon_loss = F.mse_loss(recon, orig_modalities)

        total_loss = (
            flow_loss
            + recon_loss * self.reconstruction_loss_weight
            + velocity_loss * self.velocity_consistency_loss_weight
        )

        return total_loss

#### Run Training loop

In [124]:
from adam_atan2_pytorch import MuonAdamAtan2

from einops import rearrange

import torchvision
import torchvision.transforms as T
from torchvision.utils import save_image

from accelerate import Accelerator


In [125]:
class Encoder(Module):
    def forward(self, x):
        x = rearrange(x, '... 1 (h p1) (w p2) -> ... h w (p1 p2)', p1 = 2, p2 = 2)
        return x * 2 - 1

class Decoder(Module):
    def forward(self, x):
        x = rearrange(x, '... h w (p1 p2) -> ... 1 (h p1) (w p2)', p1 = 2, p2 = 2, h = 14)
        return ((x + 1) * 0.5).clamp(min = 0., max = 1.)


In [126]:
model = Transfusion(
    num_text_tokens=10,
    dim_latent=4,
    channel_first_latent=False,
    modality_default_shape=(14, 14),
    modality_encoder=Encoder(),
    modality_decoder=Decoder(),
    add_pos_emb=True,
    modality_num_dim=2,
    velocity_consistency_loss_weight=0.1,
    reconstruction_loss_weight=0.1,
    model_output_clean=True,
    transformer=dict(dim=64, depth=4, dim_head=32, heads=8, attn_laser=True),
)

ema_model = model.create_ema()

In [127]:
ema_model.generate_modality_only

<bound method eval_decorator.<locals>.inner of Transfusion(
  (transformer): Transformer(
    (to_time_cond): Sequential(
      (0): RandomFourierEmbed()
      (1): Linear(in_features=65, out_features=256, bias=True)
      (2): SiLU()
    )
    (expand_stream): Reduce('... d -> ... s d', 'repeat', s=1)
    (reduce_stream): Reduce('... s d -> ... d', 'sum')
    (layers): ModuleList(
      (0): ModuleList(
        (0): None
        (1): AdaptiveWrapper(
          (fn): Attention(
            (to_qk): Sequential(
              (0): Linear(in_features=64, out_features=512, bias=False)
              (1): Rearrange('b n (qk h d) -> qk b h n d', qk=2, h=8)
            )
            (to_v): Sequential(
              (0): Linear(in_features=64, out_features=256, bias=False)
              (1): Rearrange('b n (h d) -> b h n d', h=8)
            )
            (to_gates): Sequential(
              (0): Linear(in_features=64, out_features=8, bias=False)
              (1): Rearrange('b n h -> b h n 1

In [128]:
class MnistDataset(Dataset):
    def __init__(self, download=False):
        self.mnist = torchvision.datasets.MNIST(
            DATA_DIRECTORY,
            download=download
        )
    
    def __len__(self):
        return len(self.mnist)
    
    def __getitem__(self, idx):
        pil, labels = self.mnist[idx]
        digit_tensor = T.PILToTensor()(pil)
        return (digit_tensor / 255).float()
    
def cycle(iter_dl):
    while True:
        for batch in iter_dl:
            yield batch

In [129]:
import os

results_path = os.path.join(RESULTS_DIRECTORY, "transfusion", "image_only")
results_folder = Path(results_path)

rmtree(results_path, ignore_errors=True)
results_folder.mkdir(exist_ok=True, parents=True)

In [130]:
dataset = MnistDataset()
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

iter_dl = cycle(dataloader)

In [131]:
optimizer = MuonAdamAtan2(model.muon_parameters(), model.parameters(), lr = 8e-4)

accelerator = Accelerator()

ema_model.to(accelerator.device)

model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

In [ ]:
for step in range(1, 100_000 + 1):
    loss = model(next(iter_dl), velocity_consistency_ema_model=ema_model)
    accelerator.backward(loss)

    accelerator.clip_grad_norm_(model.parameters(), 0.5)

    optimizer.step()
    optimizer.zero_grad()

    ema_model.update()

    accelerator.print(f"{step}: {loss.item():.3f}")

    if divisible_by(step, 500):
        accelerator.wait_for_everyone()

        if accelerator.is_main_process:
            image = ema_model.generate_modality_only(batch_size=64)

            save_image(
                rearrange(image, "(gh gw) 1 h w -> 1 (gh h) (gw w)", gh=8)
                .detach()
                .cpu(),
                str(results_folder / f"{step}.png"),
            )


1: 36.261
2: 244.932
3: 30.044
4: 315.770
5: 13.697
6: 68.144
7: 14.105
8: 5.226
9: 6.305
10: 32.297


KeyboardInterrupt: 